### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [1]:
import polars.selectors as cs
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path
folder= Path().resolve().parent /'input'
meta_data_folder=Path().resolve().parent / 'metadata'

### 1. Load multiple dataset for different buildings.

In [3]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / '*')

In [4]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [5]:
%%time
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
Meta_data=pl.scan_parquet(meta_data_folder/'MetaData.parquet').with_columns(
    pl.col('bldg_id').cast(pl.UInt32)).filter(
    pl.col('bldg_id').is_in(bldgs
                           )).select(pl.col('in.occupants').cast(pl.Int32),'in.state','in.county',
                        'in.representative_income','in.area_median_income','in.income',
                        'in.income_recs_2020','in.income_recs_2015',
                        'in.building_america_climate_zone','in.ashrae_iecc_climate_zone_2004_sub_cz_split',
                        pl.col('in.bedrooms').cast(pl.Int32),'in.tenure','in.household_has_tribal_persons','bldg_id').unique()

CPU times: user 519 μs, sys: 0 ns, total: 519 μs
Wall time: 447 μs


In [6]:
%%time
# merge the climate zone changes into the original data
df=data.join(Meta_data, on='bldg_id')

CPU times: user 44 μs, sys: 4 μs, total: 48 μs
Wall time: 49.8 μs


In [7]:
df=df.rename({'in.ashrae_iecc_climate_zone_2004_sub_cz_split': 'climatezone',
                'out.electricity.cooling.energy_consumption..kwh': 'out.electricity.AC.energy_consumption..kwh'}).with_columns(
            pl.col('timestamp').dt.weekday().alias('day of the week'),
            pl.col('timestamp').dt.hour().alias('hour of the day'),
            pl.col('timestamp').dt.day().alias('day of the month'),
            pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
            pl.col('timestamp').dt.week().alias('week of the year'),
            pl.col('timestamp').dt.month().alias('month of the year'),
            pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
            pl.when(pl.col('day of the week').is_in([6,7])).then(pl.lit("Yes")).otherwise(pl.lit("No")).alias('IsWeekend')
            ).select(
    cs.matches('^out.electricity.*|^out.site_energy.*|^bldg*|^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$')
).drop_nulls().collect()


----------------------------------------------

### Label Encoder for categorical variables

In [8]:
from sklearn.preprocessing import LabelEncoder
def encoding_categories(x):
    le=LabelEncoder()
    for dtype,col in zip(x.dtypes,x.columns):
        if dtype==pl.String:
            x=x.with_columns(pl.Series(col, le.fit_transform(x[col])))
    return x

## Preprocess the data

In [9]:
# defining x and y
def prepare_observations(d):
    x=d.select(cs.matches('^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$').exclude('in.sqft'),
            pl.col('out.electricity.total.energy_consumption..kwh').alias('total_usage')).with_columns(pl.col('timestamp').dt.timestamp())
    y=d.select(cs.matches('^out.electricity.*..kwh$').exclude('out.electricity.total.energy_consumption..kwh'))
    x=encoding_categories(x)
    return x,y

## test standardizing the data with the standard scaler

In [10]:
from sklearn.preprocessing import StandardScaler
def normlize_standard(x,y=None):
    scaler=StandardScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x, y
    else:
        return x

## test standardizing the data with the Min Max Scaler

In [11]:
from sklearn.preprocessing import MinMaxScaler
def normalize_minMax(x, y=None):
    scaler=MinMaxScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x,y 
    else:
        return x

### Prepare cross-validation model


In [12]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test, metric):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    return metric(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [13]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import shuffle
import xgboost as xg
def cross_folding(x,y, metric):
    kf=KFold(n_splits=5, shuffle=True)
    arr=[]
    lf=pd.DataFrame()
    for i, (train,test) in enumerate(kf.split(x,y)):
        x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
        LR=model(LinearRegression(), x_train, x_test, y_train, y_test, metric)
        RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test, metric)
        XG=model(xg.XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, metric)
        frame=pd.DataFrame({
        "Split Number": [i],
        "Linear Regression": [LR],
        "Random Forest": [RF],
        "XG Boost": [XG]
        })
        arr.append(frame)
    lf=pd.concat(arr)
    return lf.mean()

In [14]:
def without_preprocessing(d):
    x,y=prepare_observations(d)
    average=cross_folding(x, y, r2_score)
    return average

In [15]:
def with_standardScalar(d):
    x,y=prepare_observations(d)
    x=normlize_standard(x)
    average=cross_folding(x, y, r2_score)
    return average

In [16]:
def with_minMaxScalar(d):
    x,y=prepare_observations(d)
    x=normalize_minMax(x)
    average=cross_folding(x, y, r2_score)
    return average

In [17]:
def apply_modelling(d):
    average_0=without_preprocessing(d)
    average_1=with_standardScalar(d)
    average_2=with_minMaxScalar(d)   
    return pd.concat([average_0, average_1, average_2], axis=1, keys=['no normalization','standard Scalar','MinMaxScalar'])

In [ ]:
apply_modelling(df)

---------------------

### Tuning XGboost Model

In [ ]:
from sklearn.model_selection import GridSearchCV
def tunning(training_data, testing_data, model, param):
        x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.3)
        grid_search = GridSearchCV(model, param_grid=param, cv=KFold(3), 
                                   scoring='r2',
                                   n_jobs=-1)
        grid_search.fit(x_train, y_train)
        return grid_search.best_params_, grid_search.best_score_

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
params= {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'subsample': [0.5, 0.7, 1]
}
x,y=prepare_observations(df)
best_param, best_score=tunning(x,y, XGBRegressor(), params)

In [ ]:
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.3, random_state=1)
XG=model(XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, r2_score)
XG

### Deep Learning with pytorch

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as tf

In [19]:
import gc
torch.cuda.empty_cache()
gc.collect()

4315

In [61]:
class Model(nn.Module):
    def __init__(self, features_num=23, h1=23, h2=12, out_features=29):
        super().__init__()
        self.fc1 = nn.Linear(features_num, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_features)
    def forward(self, x):
        x=tf.relu(self.fc1(x))
        x=tf.relu(self.fc2(x))
        x=self.fc3(x)
        return x

In [62]:
from sklearn.model_selection import train_test_split
model=Model()
x,y=prepare_observations(df)
x,y=normlize_standard(x, y)
x_train, x_test, y_train, y_test=train_test_split(x, y, random_state=42, test_size=0.3)
x_train=torch.from_numpy(x_train).float()
x_test=torch.from_numpy(x_test).float()
y_train=torch.from_numpy(y_train).float()
y_test=torch.from_numpy(y_test).float()

In [63]:
## Creating a criterion to measure the error of the model
criterion=nn.MSELoss()

In [64]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [65]:
# from torch.utils.data import TensorDataset, DataLoader
# data_set=TensorDataset(x_train, y_train)
# dataset=DataLoader(dataset=data_set, batch_size=2048, shuffle=True)

In [ ]:
epochs=10000
losses=[]
for i in range(epochs):
    # for x, y in dataset:
    y_pred=model.forward(x_train)
    loss=criterion(y_pred, y_train)
    losses.append(loss)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if i %10 ==0:
        print(f"{i} iteration, loss: {loss}")

0 iteration, loss: 0.906399130821228
10 iteration, loss: 0.8848325610160828
20 iteration, loss: 0.8707179427146912
30 iteration, loss: 0.8603084087371826
40 iteration, loss: 0.85122150182724
50 iteration, loss: 0.8422245383262634
60 iteration, loss: 0.8325784802436829
70 iteration, loss: 0.8217964172363281
80 iteration, loss: 0.8096786141395569
90 iteration, loss: 0.795994222164154
100 iteration, loss: 0.780591607093811
110 iteration, loss: 0.7634520530700684
120 iteration, loss: 0.7446858882904053
130 iteration, loss: 0.7247560024261475
140 iteration, loss: 0.7045073509216309
150 iteration, loss: 0.6847338676452637
160 iteration, loss: 0.6659728288650513
170 iteration, loss: 0.6484503746032715
180 iteration, loss: 0.6321539878845215
190 iteration, loss: 0.6170306205749512
200 iteration, loss: 0.6030723452568054
210 iteration, loss: 0.5902361273765564
220 iteration, loss: 0.5784931182861328
230 iteration, loss: 0.5678226947784424
240 iteration, loss: 0.558133602142334
250 iteration, lo

In [34]:
rnn = nn.LSTM(10, 20, num_layers=2, batch_first=True)

LSTM(10, 20, num_layers=2, batch_first=True)

In [42]:
input = torch.randn(5, 3, 10) # 3D 1d=5, 2d=4, 3d=10

In [ ]:
h0 = torch.randn(2, 5, 20)
c0 = torch.randn(2, 5, 20)

In [60]:
output, (hn, cn) = rnn(input, (h0, c0))
output[:,-1,:]

tensor([[ 0.0755, -0.0825,  0.1016,  0.0238, -0.0611,  0.0517, -0.1177, -0.0414,
         -0.0971,  0.0115, -0.1107,  0.0780, -0.0711,  0.1692, -0.2363,  0.0211,
         -0.2249,  0.1720,  0.1419, -0.1125],
        [ 0.1061, -0.1289,  0.0919,  0.1071, -0.0625, -0.0196, -0.1146,  0.0185,
         -0.0447, -0.0200,  0.0690,  0.1692,  0.1660,  0.0664, -0.1529, -0.0448,
         -0.2273,  0.1460,  0.0473, -0.0464],
        [ 0.1132, -0.0201,  0.1527,  0.1109, -0.1396, -0.0436, -0.1290, -0.1968,
         -0.0669,  0.0057, -0.0877,  0.2025,  0.0975,  0.1302, -0.1916,  0.0333,
         -0.2164,  0.1496, -0.0174, -0.0432],
        [ 0.0868, -0.0771,  0.0388,  0.0956, -0.1624, -0.0005, -0.0502,  0.0316,
         -0.1299, -0.0038, -0.1424,  0.0563,  0.0905,  0.2481, -0.0340,  0.0847,
         -0.1451,  0.0819,  0.0834, -0.1119],
        [ 0.0635, -0.1571,  0.2008,  0.0830, -0.1119, -0.0024, -0.0639, -0.0797,
         -0.1946, -0.0191, -0.1606,  0.1501, -0.0319, -0.0264, -0.1410, -0.0357,
      

In [51]:
output

tensor([[[-4.3253e-02,  1.4358e-01,  1.8229e-02,  6.8481e-02,  2.0137e-01,
          -1.6101e-02, -3.5448e-01,  1.3407e-01,  2.1338e-01, -3.2954e-01,
          -4.8964e-01, -9.4488e-03, -2.4791e-01,  1.8230e-01, -4.0105e-02,
           5.1875e-02, -4.1649e-01,  3.5545e-01,  3.4562e-01, -2.2863e-02],
         [ 4.9244e-02, -1.3361e-02,  7.9511e-02,  4.8016e-02,  3.1344e-02,
           5.5957e-02, -1.5725e-01,  6.8136e-03, -3.5874e-02, -6.3120e-02,
          -2.2233e-01,  2.8720e-02, -1.5813e-01,  1.9600e-01, -2.2069e-01,
           3.8294e-02, -3.1558e-01,  2.1834e-01,  2.0059e-01, -1.0331e-01],
         [ 7.5517e-02, -8.2454e-02,  1.0164e-01,  2.3792e-02, -6.1078e-02,
           5.1704e-02, -1.1771e-01, -4.1423e-02, -9.7142e-02,  1.1465e-02,
          -1.1071e-01,  7.7965e-02, -7.1110e-02,  1.6924e-01, -2.3632e-01,
           2.1066e-02, -2.2490e-01,  1.7203e-01,  1.4194e-01, -1.1249e-01]],

        [[ 1.3753e-01, -3.3002e-01, -1.1359e-02,  5.5332e-01, -3.5474e-02,
           2.2269e-0

## Calculating measures

In [ ]:
# prepare measurement
Test_data=x.head(1)
Actual_data=y.row(1)

In [ ]:
Test_data

In [ ]:
Actual_data

In [ ]:
# Select the best model 
import xgboost as xg
from sklearn.model_selection import train_test_split
x,x_test, y, y_test= train_test_split(x,y, test_size=.3, random_state=42)
xg_model=xg.XGBRegressor()
xg_model.fit(x,y)
estimated_values=xg_model.predict(Test_data)

In [ ]:
estimated_values.flatten()

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.

Input data = timestamp Total      Usage     Climate Zone.
             1545212700000000     1.22827       1

Actual result=(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0).


Estimated result= [2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03].